## Few Shot Prompting

Here we provide a list of examples, which are converted to prompt using template + Make the template using FewShotPromptTemplate
- Suffix is provided, for the end of the prompt, where we define our own input

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

In [5]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(
    base_url=BASE_URL, 
    api_key=API_KEY, 
    model='gpt-5.1',
    temperature=1.4
)

In [6]:
examples = [
    {
        "input": "A train leaves city A for city B at 60 km/h, and another train leaves city B for city A at 40 km/h. If the distance between them is 200 km, how long until they meet?", 
        "thought": "The trains are moving towards each other, so their relative speed is 60 + 40 = 100 km/h. The time to meet is distance divided by relative speed: 200 / 100 = 2 hours.",
        "output": "2 hours",
    },
    {
        "input": "If a store applies a 20% discount to a $50 item, what is the final price?", 
        "thought": "A 20% discount means multiplying by 0.8. So, $50 × 0.8 = $40.",
        "output": "$40",
    },
    {
        "input": "A farmer has chickens and cows. If there are 10 heads and 32 legs, how many of each animal are there?", 
        "thought": "Let x be chickens and y be cows. We have two equations: x + y = 10 (heads) and 2x + 4y = 32 (legs). Solving: x + y = 10 → x = 10 - y. Substituting: 2(10 - y) + 4y = 32 → 20 - 2y + 4y = 32 → 2y = 12 → y = 6, so x = 4.",
        "output": "4 chickens, 6 cows",
    },
    {
        "input": "If a car travels 90 km in 1.5 hours, what is its average speed?", 
        "thought": "Speed is distance divided by time: 90 km / 1.5 hours = 60 km/h.",
        "output": "60 km/h",
    },
    {
        "input": "John is twice as old as Alice. In 5 years, their combined age will be 35. How old is Alice now?", 
        "thought": "Let Alice's age be x. Then John’s age is 2x. In 5 years, their ages will be x+5 and 2x+5. Their sum is 35: x+5 + 2x+5 = 35 → 3x + 10 = 35 → 3x = 25 → x = 8.33.",
        "output": "8.33 years old",
    },
]

In [11]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

example_prompt = PromptTemplate(template = 'Question: {input}\nThought: {thought}\nAnswer: {output}')
example= example_prompt.invoke(examples[0])
example

StringPromptValue(text='Question: A train leaves city A for city B at 60 km/h, and another train leaves city B for city A at 40 km/h. If the distance between them is 200 km, how long until they meet?\nThought: The trains are moving towards each other, so their relative speed is 60 + 40 = 100 km/h. The time to meet is distance divided by relative speed: 200 / 100 = 2 hours.\nAnswer: 2 hours')

In [14]:
prompt_template = FewShotPromptTemplate(
    examples = examples,
    example_prompt = example_prompt,
    suffix = 'Question: {input}'
)

prompt_template

FewShotPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, examples=[{'input': 'A train leaves city A for city B at 60 km/h, and another train leaves city B for city A at 40 km/h. If the distance between them is 200 km, how long until they meet?', 'thought': 'The trains are moving towards each other, so their relative speed is 60 + 40 = 100 km/h. The time to meet is distance divided by relative speed: 200 / 100 = 2 hours.', 'output': '2 hours'}, {'input': 'If a store applies a 20% discount to a $50 item, what is the final price?', 'thought': 'A 20% discount means multiplying by 0.8. So, $50 × 0.8 = $40.', 'output': '$40'}, {'input': 'A farmer has chickens and cows. If there are 10 heads and 32 legs, how many of each animal are there?', 'thought': 'Let x be chickens and y be cows. We have two equations: x + y = 10 (heads) and 2x + 4y = 32 (legs). Solving: x + y = 10 → x = 10 - y. Substituting: 2(10 - y) + 4y = 32 → 20 - 2y + 4y = 32 → 2y = 12 → y = 6, so x = 

In [15]:
chain = prompt_template | model
result = chain.invoke({'input': 'What will be 3 days after tomorrow, if today is Thrusday?'})
print(result.content)

Today is Thursday.

- Tomorrow: Friday  
- 3 days after tomorrow: Saturday, Sunday, **Monday**

Answer: Monday
